<a href="https://colab.research.google.com/github/GiX007/weather_time_series_forecasting/blob/master/auxiliary/tsf_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises in Weather Time-Series Forecasting

A collection of some intuitions with solutions.

In [38]:
import numpy as np
import torch

## Sliding windows for multi-horizon forecasting

In [2]:
def make_windows(y, L, H):
  """
  Turns a 1D array y into input windows of length L and multi-step targets of length H.
  """
  N = len(y) - L - H + 1 # number of samples
  X = np.stack([y[i:i+L] for i in range(N)]) # (N, L)
  Y = np.stack([y[i+L:i+L+H] for i in range(N)]) # (N, H)
  return X, Y

In [3]:
y = [1, 2, 3, 4, 5, 6]
X, Y = make_windows(y, L=3, H=2)
print(X.tolist()) # [[[1],[2],[3]], [[2],[3],[4]]]
print(Y.tolist()) # [[4,5], [5,6]]

[[1, 2, 3], [2, 3, 4]]
[[4, 5], [5, 6]]


## Recursive rolling forecast (closed loop)

In [30]:
@torch.no_grad()
def recursive_rollout(model, x0, n_steps):
  """
  Implements a recursive rollout: feeds each prediction back into the model to make a new prediction.
  """
  # x0: (B, L, 1) tensor
  preds = []
  x = x0.clone()
  for _ in range(n_steps):
    y1 = model(x) # (B, 1, 1)
    y1_flat = y1.squeeze(-1).squeeze(-1) # (B,)
    preds.append(y1_flat) # (B,)
    x = torch.cat([x[:, 1:, :], y1], dim=1) # (B, L, 1)
  return torch.stack(preds, dim=1) # (B, n_steps)

In [31]:
class DummyModel(torch.nn.Module):
  def forward(self, x):
    return x[:, -1:, :] + 1 # just predicts "last +1"

In [32]:
x0 = torch.tensor([[[1],[2],[3]]], dtype=torch.float)
x0, x0.shape

(tensor([[[1.],
          [2.],
          [3.]]]),
 torch.Size([1, 3, 1]))

In [34]:
preds = recursive_rollout(DummyModel(), x0, n_steps=3)
print(preds.shape, preds) # [1, 3], [[4,5,6]]

torch.Size([1, 3]) tensor([[4., 5., 6.]])


## Teacher-forcing rolling forecast (uses true next value)

In [68]:
@torch.no_grad()
def teacher_forcing_rollout(model, x0, y_future):
  """
  Implements teacher forcing: at each step, uses the TRUE next value to update the window instead of the model's prediction.
  """
  # x0: (B, L, 1) tensor
  # y_future: (B, n_steps) tensor of ground-truth next values
  preds = []
  x = x0.clone()
  for i in range(y_future.shape[1]):
    y1 = model(x) # (B, 1, 1)
    y1_flat = y1.squeeze(-1).squeeze(-1) # (B,)
    preds.append(y1_flat) # (B,)

    true_next = y_future[:, i] # (B,)
    # feed the TRUE next value back into the window
    x = torch.cat([x[:, 1:, :], true_next.unsqueeze(-1).unsqueeze(-1)], dim=1) # (B, L, 1)

  return torch.stack(preds, dim=1) # (B, n_steps)

In [69]:
y_future = torch.tensor([[4,5,6]], dtype=torch.float)
preds_tf = teacher_forcing_rollout(DummyModel(), x0, y_future)
print(preds_tf.shape, preds_tf) # [1, 3], [[4,5,6]]

torch.Size([1, 3]) tensor([[4., 5., 6.]])


## Convertion of normalized predicted diffs back to raw levels

In [35]:
def diffs_to_levels(pred_diffs_norm, last_level, mu_diff, sigma_diff):
  """
  Given mu, sigma of the diff series and the last level, it converts the normalized predicted diffs back to raw levels.
  """
  diffs_raw = pred_diffs_norm * sigma_diff + mu_diff
  levels = []
  base = last_level
  for d in diffs_raw:
    base = base + d
    levels.append(base)
  return np.array(levels)

In [37]:
pred_diffs_norm = np.array([0.0, 1.0]) # normalized
last_level = 10
mu_diff, sigma_diff = 2, 1
print(diffs_to_levels(pred_diffs_norm, last_level, mu_diff, sigma_diff)) # [12, 15]

[12. 15.]


## Tiny Transformer-Encoder layer

In [45]:
class TinyTransformerEncoder(torch.nn.Module):
  def __init__(self, d_model=8, n_heads=2, ffn_dim=128, p=0.1):
    super().__init__()
    self.ln1 = torch.nn.LayerNorm(d_model)
    self.mha = torch.nn.MultiheadAttention(d_model, n_heads, dropout=p, batch_first=True)
    self.dropout = torch.nn.Dropout(p)
    self.ln2 = torch.nn.LayerNorm(d_model)
    self.ffn = torch.nn.Sequential(
      torch.nn.Linear(d_model, ffn_dim),
      torch.nn.ReLU(),
      torch.nn.Dropout(p),
      torch.nn.Linear(ffn_dim, d_model),
    )
    self.dropout2 = torch.nn.Dropout(p)

  def forward(self, x):
    # x: (B, L, d_model)
    h = self.ln1(x) # (B, L, d_model)
    att_out, _ = self.mha(h, h, h) # (B, L, d_model)
    x = x + self.dropout(att_out) # (B, L, d_model)
    h = self.ln2(x) # (B, L, d_model)
    ffn_out = self.ffn(h) # (B, L, d_model)
    x = x + self.dropout2(ffn_out) # (B, L, d_model)
    return x

In [49]:
layer = TinyTransformerEncoder()
x = torch.randn(2, 5, 8)
print(x)

tensor([[[-1.3100,  0.3710, -1.0038, -1.3274,  1.4384,  0.7284, -0.3265,
          -0.4805],
         [-0.5326,  0.5577,  0.7203,  0.4603, -0.2854, -1.4004, -0.8482,
           0.8299],
         [ 1.1236, -1.6731,  0.4017,  0.2802,  0.4458, -0.4501, -0.0142,
          -0.3012],
         [ 1.1986,  0.3107,  0.2618, -1.0292, -0.2148, -1.0856, -0.6097,
           0.4055],
         [-1.7830,  0.4184,  1.8561, -1.6243,  0.3666,  0.8002,  0.4882,
          -1.5592]],

        [[ 0.3816, -1.6723, -0.5081,  1.0826, -0.9416, -0.3210, -0.3847,
          -1.1190],
         [-0.4559,  0.6394,  0.3961, -3.0748, -0.2029, -0.2336, -0.0034,
           0.0571],
         [-0.6783,  0.1345,  0.0185, -0.2775, -1.0791,  0.3653, -1.0465,
          -0.2360],
         [-1.4745,  0.4643,  2.0687, -0.0038, -1.6811, -0.8935,  0.1507,
          -0.3348],
         [ 0.0832,  0.2387,  0.4184, -1.0642, -1.0577,  0.8136, -0.2144,
           0.7574]]])


In [50]:
out = layer(x)
print(out.shape)  # (2, 5, 8)

torch.Size([2, 5, 8])


In [51]:
print(out)

tensor([[[-1.5584,  0.5187, -0.6615, -1.0563,  0.8617,  0.6355, -0.4669,
          -0.5641],
         [-0.1653,  0.9975,  0.8524, -0.1874, -0.1158, -1.3003, -1.0248,
           0.7332],
         [ 1.5577, -2.1206,  0.6664,  0.1062,  0.8676, -0.1235, -0.1309,
          -0.3539],
         [ 1.2209,  0.6917,  0.6340, -1.5222, -0.4426, -1.3056, -1.2589,
           0.3264],
         [-2.1395,  0.7446,  2.0229, -1.2006,  0.0250,  0.5213, -0.0818,
          -1.8689]],

        [[ 0.5110, -1.9965, -0.3617,  1.2685, -0.8599, -0.0349, -0.4789,
          -0.9151],
         [-0.9905,  0.9356,  0.5576, -2.8404, -0.2438, -0.3533,  0.1611,
           0.2922],
         [-0.7173, -0.1930,  0.1597, -0.2939, -0.8313,  0.2862, -0.9439,
           0.0301],
         [-1.5708,  0.4630,  2.2653, -0.7450, -1.3182, -0.6019, -0.0310,
           0.0458],
         [-0.4043,  0.1097,  0.6950, -0.8500, -1.0450,  0.7801, -0.3308,
           1.2323]]], grad_fn=<AddBackward0>)
